In [ ]:
import os
import shutil
from prettytable import PrettyTable

from pyspark import SparkContext, SparkConf

# 1. Initialize Spark Context (Low-level RDD API)
conf = SparkConf().setAppName("RDDBucketingTest").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)

for path in ["partitioned_users_disk", "partitioned_orders_disk", "users_disk", "orders_disk", "partitioned_join", "standard_join"]:
    if os.path.exists(path):
        shutil.rmtree(path)

print("--- Generating Mock RDD Data ---")
# Create raw data: (user_id, name) and (user_id, order_amount)
users_raw = [(1, "Alice"), (2, "Bob"), (3, "Charlie"), (4, "David")]
orders_raw = [(1, 10.50), (2, 20.00), (1, 55.00), (3, 99.99), (4, 4.50)]

# Load them into standard, unpartitioned RDDs
users_rdd = sc.parallelize(users_raw)
orders_rdd = sc.parallelize(orders_raw)

users_rdd.saveAsTextFile("users_disk")
orders_rdd.saveAsTextFile("orders_disk")



# TEST 1: STANDARD RDD JOIN

print("TEST 1: Standard RDD Join")
standard_join = users_rdd.join(orders_rdd)

standard_join.saveAsTextFile("standard_join")
print(standard_join.collect())
print(standard_join.toDebugString().decode())



# TEST 2: PARTITIONED RDD JOIN
print("\nTEST 2: Partitioned RDD Join")


# Apply the partitioner to both RDDs and cache them to lock the structure in memory
partitioned_users = users_rdd.partitionBy(4).cache()
partitioned_orders = orders_rdd.partitionBy(4).cache()

partitioned_users.saveAsTextFile("partitioned_users_disk")
partitioned_orders.saveAsTextFile("partitioned_orders_disk")

# Run the exact same join on the pre-partitioned RDDs
partitioned_join = partitioned_users.join(partitioned_orders)
print(partitioned_join.toDebugString().decode())

partitioned_join.saveAsTextFile("partitioned_join")

print(partitioned_join.collect())




--- Generating Mock RDD Data ---
TEST 1: Standard RDD Join


[(1, ('Alice', 10.5)), (1, ('Alice', 55.0)), (2, ('Bob', 20.0)), (3, ('Charlie', 99.99)), (4, ('David', 4.5))]
(48) PythonRDD[365] at collect at /tmp/ipykernel_18047/2197409829.py:35 []
 |   MapPartitionsRDD[361] at mapPartitions at PythonRDD.scala:170 []
 |   ShuffledRDD[360] at partitionBy at <unknown>:0 []
 +-(48) PairwiseRDD[359] at join at /tmp/ipykernel_18047/2197409829.py:32 []
    |   PythonRDD[358] at join at /tmp/ipykernel_18047/2197409829.py:32 []
    |   UnionRDD[357] at union at <unknown>:0 []
    |   PythonRDD[355] at RDD at PythonRDD.scala:58 []
    |   ParallelCollectionRDD[347] at readRDDFromFile at PythonRDD.scala:299 []
    |   PythonRDD[356] at RDD at PythonRDD.scala:58 []
    |   ParallelCollectionRDD[348] at readRDDFromFile at PythonRDD.scala:299 []

TEST 2: Partitioned RDD Join
(4) PythonRDD[383] at RDD at PythonRDD.scala:58 []
 |  PartitionerAwareUnionRDD[382] at union at <unknown>:0 []
 |  PythonRDD[380] at RDD at PythonRDD.scala:58 []
 |  MapPartitionsRDD[369]